# 02 · Índice de consumo energético: provincias

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/santiagoriverti/consumo_energetico_argentina/blob/main/notebooks/02_provincias.ipynb)

Distribución geográfica del consumo energético industrial (electricidad + gas, en TJ) y su evolución.
Para evitar la estacionalidad, todo se compara en **sumas de 12 meses**.

- Buenos Aires incluye CABA (CAMMESA no las separa).
- Tierra del Fuego no está conectada al sistema eléctrico nacional (solo gas); Misiones y Formosa no tienen
  red de gas natural (solo electricidad).
- El *by-pass* industrial de gas (industrias conectadas directo a los gasoductos) se reparte entre las
  provincias de cada área de licencia según el consumo industrial de distribución del mes. En la Patagonia
  (área de Camuzzi Gas del Sur) ese reparto pesa mucho: tomar con cuidado las variaciones de Neuquén, Río
  Negro, Chubut y Santa Cruz.

In [ ]:
# Configuracion: en Colab clona (o actualiza) el repo; en local usa la carpeta del repo
import os, sys, subprocess, importlib
REPO = 'https://github.com/santiagoriverti/consumo_energetico_argentina.git'
if 'google.colab' in sys.modules:
    RAIZ = '/content/consumo_energetico_argentina'
    if os.path.exists(RAIZ):
        # Trae la ultima version de GitHub y descarta cambios locales (la copia de Colab es descartable)
        subprocess.run(['git', '-C', RAIZ, 'fetch', '-q', '--depth', '1', 'origin', 'main'], check=True)
        subprocess.run(['git', '-C', RAIZ, 'reset', '-q', '--hard', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, RAIZ], check=True)
else:  # subir desde la carpeta actual hasta encontrar el repo
    RAIZ = os.getcwd()
    while not os.path.exists(os.path.join(RAIZ, 'src', 'pipeline.py')) and os.path.dirname(RAIZ) != RAIZ:
        RAIZ = os.path.dirname(RAIZ)
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
# Si el notebook ya se corrio en esta sesion, olvidar los modulos de src cargados (pueden ser viejos)
for m in [m for m in sys.modules if m == 'src' or m.startswith('src.')]:
    del sys.modules[m]
importlib.invalidate_caches()
v = subprocess.run(['git', '-C', RAIZ, 'log', '-1', '--format=%h (%ad) %s', '--date=short'],
                   capture_output=True, text=True).stdout.strip()
print('Version del repo:', v or 'sin git')

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from src import pipeline, graficos, indices
pd.set_option('display.float_format', lambda x: f'{x:,.1f}')
pd.set_option('display.max_columns', 30)

In [ ]:
# ACTUALIZAR = True vuelve a bajar CAMMESA y ENARGAS (~100 MB, unos minutos). Con False usa las tablas
# del repo (data/processed/), que se actualizan con scripts/construir.py.
ACTUALIZAR = False
R = pipeline.calcular(actualizar=ACTUALIZAR)
print(R['meta'])

## Dónde se consume la energía industrial

In [ ]:
fig = graficos.g07_ranking_provincias(R); plt.show()

In [ ]:
t = R['provincias_industria']
display(t[['region', 'tj_12m', 'participacion_%', 'part_electricidad_%', 'var_12m_vs_anterior_%',
           'var_12m_vs_anterior_electricidad_%', 'var_12m_vs_anterior_gas_%', 'var_12m_vs_2023_%',
           f'var_12m_vs_{indices.BASE}_%']])

## Cómo cambió desde 2023

Las provincias chicas pueden mostrar variaciones grandes por un solo usuario: leer el mapa junto con la
participación de cada provincia en el total (tabla de arriba).

In [ ]:
fig = graficos.g06_mapa_industria(R, 'var_12m_vs_2023_%'); plt.show()

In [ ]:
fig = graficos.g06_mapa_industria(R, 'var_12m_vs_anterior_%'); plt.show()

## Trayectoria por provincia

Suma de 12 meses a diciembre de cada año (y al último mes) con 2016 = 100.

In [ ]:
fig = graficos.g08_heatmap_provincias(R); plt.show()

## Por región

In [ ]:
from src import procesar
p = R['panel']
p = p[(p['sector'] == 'industria') & (p['fecha'] <= pd.Timestamp(R['meta']['ultimo_mes_indice']))]
reg = p.merge(procesar.provincias()[['provincia', 'region']], on='provincia')
anual = reg.groupby([reg['fecha'].dt.year, 'region'])['tj'].sum().unstack()
anual = anual[reg.groupby(reg['fecha'].dt.year)['fecha'].nunique() == 12]
display((100 * anual / anual.loc[indices.BASE]).round(1))

## Hogares por provincia

In [ ]:
display(R['provincias_hogares'][['region', 'tj_12m', 'participacion_%', 'part_electricidad_%',
                                   'var_12m_vs_anterior_%', 'var_12m_vs_2023_%']])

Gas por usuario residencial (m³ por mes, promedio de 12 meses): el clima de cada provincia pesa más que
cualquier otra cosa.

In [ ]:
g = R['gas_por_usuario'].rolling(12).mean()
t = g.iloc[[-37, -13, -1]].T
t.columns = [d.strftime('12m a %m/%Y') for d in t.columns]
display(t.sort_values(t.columns[-1], ascending=False))

## Descargar todos los resultados

ZIP con el Excel (una hoja por tabla), los gráficos en PNG (`graficos/`) y las tablas en CSV
(`datos/`). En Colab se descarga solo; en la PC queda en la carpeta `_descargas/` del repo.

In [ ]:
from src.exportar import zip_resultados
zip_path = zip_resultados(R, 'ICE_02_provincias')
print('ZIP:', zip_path)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(zip_path)